# IndicOOC — Zero-Shot Baseline (Objective O2-i)

This notebook takes your `pilot_v0.csv` and runs it through the same SigLIP mechanism you smoke-tested in `IndicOOC_00_Setup.ipynb` — but now on real (image, claim, label) triples instead of the cats/fire-truck toy example.

**What it does, in order:**
1. Load the pilot data and apply corrections for two known schema issues (language mislabel, out-of-scope manipulated instance)
2. Download each image, failing gracefully instead of crashing on link rot
3. Score every (image, claim) pair with SigLIP and threshold it into `supported` / `out_of_context`
4. Report macro-F1 overall and broken down by claim language
5. Check the matched pairs specifically — does the model at least rank the true caption above the false one for the *same* image?

**Deliberate choice, not an oversight:** this uses the same English-trained `google/siglip-base-patch16-224` checkpoint as the setup notebook, not a multilingual variant. Testing whether an English-only encoder transfers to Devanagari/Bengali/Marathi script *is* the experiment your synopsis calls for in Section 5.2 — if it performs badly on non-English claims, that's a result, not a bug.

## Known data caveats (from your own notes in pilot_v0.csv)

This notebook handles three of the four flagged issues explicitly, in code, rather than silently ignoring them:

| id | Issue | How this notebook handles it |
|---|---|---|
| `hi_ooc_001` | image_url is Alt News' composite feature card, not the raw viral image | Scored anyway (pipeline shouldn't crash), but tagged with a caveat so you don't over-read the result |
| `hi_ooc_002` | claim actually circulated in Marathi; `language` column records the *article's* language | Corrected via a `claim_language` override for grouping/metrics |
| `mr_ooc_002` | face-composite / manipulated image, not genuine out-of-context — out of scope per Section 9.2 | Excluded from the quantitative baseline, kept aside for the O4 failure taxonomy |
| Marathi has no second class after the exclusion above | Can't compute a real macro-F1 for `mr` yet | Reported as single-class accuracy with an explicit note, not a fabricated F1 |

None of this changes your CSV — the corrections live here, in the analysis layer, where they're visible and easy to undo once the real data fixes land.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers accelerate pillow pandas scikit-learn sentencepiece
print("Done.")

## Step 1 — Load the pilot data and apply the corrections

Update `DATA_PATH` if your CSV lives somewhere else in Drive. If you haven't moved `pilot_v0.csv` into the project folder yet, do that first — this expects it at `data/annotations/pilot_v0.csv`.

In [ ]:
import pandas as pd

DATA_PATH = '/content/drive/MyDrive/IndicOOC/data/annotations/pilot_v0.csv'
df = pd.read_csv(DATA_PATH)

# hi_ooc_002: the claim that actually circulated was Marathi, even though this
# fact-check was published on Alt News HINDI. `language` records the article's
# language, not the claim's — a schema gap, not a data error. Corrected here
# for analysis only; add a dedicated `claim_language` column for the real O1 pass.
LANGUAGE_OVERRIDES = {
    'hi_ooc_002': 'mr',
}
df['claim_language'] = df.apply(
    lambda r: LANGUAGE_OVERRIDES.get(r['id'], r['language']), axis=1
)

# mr_ooc_002: flagged as a face-composite (manipulated image), not a genuine
# out-of-context pairing. Deepfake/manipulation detection is explicitly out of
# scope (Section 9.2) — this tests a different phenomenon than SigLIP
# similarity thresholding is meant to catch, so it's excluded from scoring
# and set aside for the O4 failure-mode taxonomy instead.
EXCLUDE_IDS = {'mr_ooc_002'}
excluded_df = df[df['id'].isin(EXCLUDE_IDS)].copy()
eval_df = df[~df['id'].isin(EXCLUDE_IDS)].copy()

# hi_ooc_001: image_url points to a composite feature card, not the raw viral
# image. Still scored so the pipeline doesn't break, but flagged.
KNOWN_IMAGE_ISSUES = {
    'hi_ooc_001': 'image_url is a composite feature card, not the raw viral image',
}
eval_df['image_caveat'] = eval_df['id'].map(KNOWN_IMAGE_ISSUES)

print(f"Loaded {len(df)} rows -> evaluating {len(eval_df)}, excluding {len(excluded_df)} (flagged for O4)\n")
print("Claim-language counts (corrected):")
print(eval_df['claim_language'].value_counts())
print("\nLabel counts:")
print(eval_df['label'].value_counts())

## Step 2 — Robust image loading

Given the CDN link-rot risk already flagged in your notes (`mr_ooc_001`), this needs to skip failed downloads and log them, not crash the whole loop.

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def load_image(url, timeout=15):
    try:
        resp = requests.get(url, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'})
        resp.raise_for_status()
        return Image.open(BytesIO(resp.content)).convert('RGB')
    except Exception as e:
        print(f"  [skip] failed to load {url}: {e}")
        return None

## Step 3 — Load SigLIP (same checkpoint as the setup notebook)

In [ ]:
from transformers import AutoProcessor, AutoModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/siglip-base-patch16-224"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()
print(f"Loaded {model_name} on {device}")

## Step 4 — Score every pair

SigLIP's sigmoid output is an independent match probability for the (image, claim) pair — not a comparison across candidates — so we threshold it directly: above the threshold means the claim is judged consistent with the image (`supported`), below means `out_of_context`.

In [ ]:
THRESHOLD = 0.5
results = []

for _, row in eval_df.iterrows():
    image = load_image(row['image_url'])
    if image is None:
        results.append({**row.to_dict(), 'pred_label': None, 'score': None})
        continue

    inputs = processor(text=[row['claim_text']], images=image,
                        padding="max_length", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        score = torch.sigmoid(outputs.logits_per_image)[0][0].item()

    pred_label = 'supported' if score > THRESHOLD else 'out_of_context'
    results.append({**row.to_dict(), 'pred_label': pred_label, 'score': score})

results_df = pd.DataFrame(results)
results_df[['id', 'claim_language', 'label', 'pred_label', 'score', 'image_caveat']]

## Step 5 — Metrics: overall and by claim language

In [ ]:
from sklearn.metrics import classification_report, f1_score

scored = results_df.dropna(subset=['pred_label'])
n_skipped = len(results_df) - len(scored)
print(f"Scored {len(scored)}/{len(results_df)} rows ({n_skipped} skipped on failed downloads)\n")

print("=== Overall ===")
print(classification_report(scored['label'], scored['pred_label'], zero_division=0))

print("=== By claim language ===")
for lang, group in scored.groupby('claim_language'):
    if group['label'].nunique() < 2:
        acc = (group['label'] == group['pred_label']).mean()
        print(f"{lang}: only one class present ({group['label'].unique()[0]}) — "
              f"macro-F1 not meaningful yet. Raw accuracy: {acc:.2f} (n={len(group)})")
    else:
        f1 = f1_score(group['label'], group['pred_label'], average='macro', zero_division=0)
        print(f"{lang}: macro-F1 = {f1:.3f} (n={len(group)})")

## Step 6 — Bonus: matched-pair ranking check

Your minimal pairs (same image, true vs. false caption) give a cleaner signal than absolute thresholding: does the model at least *rank* the true caption above the false one for the same image, regardless of where 0.5 happens to fall?

In [ ]:
pair_ok, pair_total = 0, 0
for image_url, group in scored.groupby('image_url'):
    labels = set(group['label'])
    if labels == {'supported', 'out_of_context'}:
        sup_score = group[group['label'] == 'supported']['score'].iloc[0]
        ooc_score = group[group['label'] == 'out_of_context']['score'].iloc[0]
        pair_total += 1
        correct = sup_score > ooc_score
        pair_ok += int(correct)
        short_url = image_url.split('/')[-1]
        mark = 'correct' if correct else 'WRONG'
        print(f"{short_url}: supported={sup_score:.3f} vs out_of_context={ooc_score:.3f} [{mark}]")

if pair_total:
    print(f"\nCorrectly ranked {pair_ok}/{pair_total} matched pairs")
else:
    print("No complete matched pairs found in the scored set.")

## What this gives you, and what's still missing

This is your first real macro-F1 number(s) — genuinely earlier than most Review 1 checkpoints get one. Before trusting these numbers beyond "does the pipeline work":

- Re-scrape the real viral image for `hi_ooc_001` — the composite-card result isn't a fair test
- Add at least one Marathi `supported` example so `mr` gets a real macro-F1, not just accuracy
- 10 rows is nowhere near enough to trust any single number — treat this as "the wiring works," not as a result to report
- Once the pilot grows past ~50-100 rows, it's worth adding `google/siglip-base-patch16-256-multilingual` as a second baseline point alongside this English-only one — that comparison is a direct test of the transfer-assumption question in Section 5.2, rather than an assumption either way